# Variante Kaggle — J4

Importer ce notebook dans Kaggle. Ajouter les données officielles **playground-series-s6e11** via Add Input après acceptation personnelle des règles. Choisir CPU pour reproduire le budget local. Lancer les cellules dans l’ordre, puis conserver les sorties de `/kaggle/working/MyLittleMLLab/runs/J4`. Les numéros de cellules du PowerPoint concernent la version locale ; cette variante ajoute deux cellules d’amorçage.

Aucun secret ni token n’est inclus. La métrique et les folds restent identiques. Un dépôt éventuel se fait manuellement. Vérifier les règles, les limites du compte et l’ouverture du concours avant la séance.


In [ ]:
# Adaptateur Kaggle : mêmes cellules métier, mêmes données et mêmes folds.
from pathlib import Path
import os,base64,io,zipfile,shutil
out=Path(os.environ.get('MLLAB_KAGGLE_OUTPUT','/kaggle/working'))/'MyLittleMLLab'
out.mkdir(parents=True,exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))).extractall(out)
inputs=Path(os.environ.get('MLLAB_KAGGLE_INPUT','/kaggle/input'))
slug='playground-series-s6e11'
matches=[p.parent for p in inputs.rglob('sample_submission.csv') if slug in str(p.parent) and (p.parent/'train.csv').exists() and (p.parent/'test.csv').exists()]
if len(matches)!=1:raise FileNotFoundError('Ajouter les données officielles playground-series-s6e11 via Add Input. Un seul jeu correspondant doit être présent.')
dest=out/'data/competitions'/slug;dest.mkdir(parents=True,exist_ok=True)
for name in ['train.csv','test.csv','sample_submission.csv']:
 shutil.copy2(matches[0]/name,dest/name)
os.chdir(out)
print('Laboratoire préparé dans le répertoire de travail Kaggle ; aucun téléchargement ou dépôt automatique.')


# AML_04 — Comprendre et entraîner un réseau neuronal

**QUID :** comment les poids apprennent-ils, et le réseau mérite-t-il sa place face aux arbres ?

Même compétition, mêmes données officielles et mêmes folds que J2. Les scores sont locaux ; aucune soumission automatique.

**Parcours renforcé.** Ouvrir cette nouvelle version au début de la séance et redémarrer le noyau. Les numéros de cellules ont changé par rapport aux premiers supports.

Chaque numéro « cellule » désigne la position physique dans ce notebook, en comptant les textes. Exécuter dans l’ordre. Installation conservée depuis le ZIP initial ; aucun git pull.

Avant la séance : dans une cellule temporaire de Jupyter, exécuter `%pip install "torch>=2.4,<3"`, puis redémarrer le noyau. PyTorch utilise le CPU dans cet atelier ; aucun GPU requis.

[Guide de démarrage](https://github.com/MathieuDWeill/ai-course-student/blob/main/usecase/MyLittleMLLab/DEMARRER_J4.md) · [Cours rédigé](https://github.com/MathieuDWeill/ai-course-student/blob/main/usecase/MyLittleMLLab/lectures/AML_04_cours.md)


## Le contrat et les ressources

PyTorch sur CPU. Installer avant la séance ou utiliser la variante Kaggle. Les petites données pédagogiques sont distinctes de la fraude officielle.

In [ ]:
from pathlib import Path
import sys, json, time, hashlib, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE=Path.cwd()
ROOT=next((p if (p/'mllab/competition.py').exists() else p/'student') for p in [HERE,*HERE.parents] if (p/'mllab/competition.py').exists() or (p/'student/mllab/competition.py').exists())
sys.path.insert(0,str(ROOT))
from mllab.competition import subsets, make_model, xy, SPECS
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
LIMIT=int(os.environ.get('MLLAB_LIMIT','15000'))
tr,va,test,sample,protocol_record=subsets('fraud',root=ROOT,limit=LIMIT)
X,y=xy(tr,'fraud'),tr.is_fraud.astype(int)
Xv,yv=xy(va,'fraud'),va.is_fraud.astype(int)
folds=list(StratifiedKFold(3,shuffle=True,random_state=2026).split(X,y))
assert not set(tr.id)&set(va.id)
RUN=ROOT/'runs/J4';RUN.mkdir(parents=True,exist_ok=True)
ids_sha=hashlib.sha256(tr.id.to_numpy().tobytes()).hexdigest()
print('Train :',len(tr),'fraudes :',int(y.sum()),'| contrôle local :',len(va))
print('3 folds fixes ; ROC-AUC ; CPU ; même protocole que J2.')
print('Le contrôle local a déjà été consulté en J2 : il ne constitue plus un test vierge.')

try:
 import torch
 from torch import nn
except ImportError as exc:
 raise ImportError('Installer dans une cellule temporaire : %pip install "torch>=2.4,<3", puis redémarrer le noyau.') from exc
torch.set_num_threads(2)
print('PyTorch',torch.__version__,'; appareil CPU explicite.')
plt.rcParams.update({'figure.figsize':(10,4),'font.size':14})


## Un poids et deux tailles de pas

Calculer prédiction, perte, gradient et nouvelle perte à la main avant de lancer.

In [ ]:
w=torch.tensor(2.,requires_grad=True)
loss=(3*w-1)**2
loss.backward()
print('Avant step : poids =',w.item(),'perte =',loss.item(),'gradient =',w.grad.item())
assert w.item()==2 and w.grad.item()==30
for lr in [.01,1.]:
 new_w=2-lr*w.grad.item()
 print('Learning rate',lr,'→ poids',new_w,'→ perte',(3*new_w-1)**2)

## Les rôles de backward et step

Observer les valeurs avant et après la mise à jour et l’accumulation de gradients.

In [ ]:
w=nn.Parameter(torch.tensor(2.));optimizer=torch.optim.SGD([w],lr=.01)
optimizer.zero_grad();pred=3*w;loss=(pred-1)**2
loss.backward();before=w.detach().item();gradient=w.grad.item()
optimizer.step()
print('Avant :',before,'gradient :',gradient,'après :',w.item())
assert np.isclose(w.item(),1.7)
# Deux backward sans remise à zéro additionnent les gradients.
w2=torch.tensor(2.,requires_grad=True)
(3*w2-1).pow(2).backward();(3*w2-1).pow(2).backward()
print('Gradient accumulé sans zero_grad :',w2.grad.item())

## Un batch visible, une couche visible

Trois lignes, deux variables, deux sorties. Refaire la première ligne à la main.

In [ ]:
# Trois observations pédagogiques et deux variables. Aucun score de compétition ici.
xb=torch.tensor([[1.,0.],[0.,1.],[1.,1.]])
yb=torch.tensor([0,1,1],dtype=torch.long)
layer=nn.Linear(2,2)
with torch.no_grad():
 layer.weight.copy_(torch.tensor([[.2,-.1],[-.3,.4]]));layer.bias.zero_()
z=layer(xb)
print('X :',tuple(xb.shape),'W :',tuple(layer.weight.shape),'logits :',tuple(z.shape))
display(pd.DataFrame(z.detach().numpy(),columns=['logit_0','logit_1']))
assert torch.allclose(z,xb@layer.weight.T+layer.bias)
print('Logits de la première ligne calculés à la main : 0,2 et −0,3.')

## Logits, softmax et cross-entropy

La perte reçoit les logits. Softmax permet de lire les probabilités pour l’interprétation.

In [ ]:
pb=torch.softmax(z,dim=1)
criterion=nn.CrossEntropyLoss()
loss_b=criterion(z,yb)
display(pd.DataFrame({'classe':yb.numpy(),'proba_fraude':pb[:,1].detach().numpy()}))
print('Perte moyenne :',loss_b.item())
manual=-torch.log(pb[torch.arange(len(yb)),yb]).mean()
assert torch.allclose(loss_b,manual)
print('Vérification : moyenne de −log(probabilité de la vraie classe) =',manual.item())

## Les gradients de chaque poids

Backward calcule les gradients sans modifier les poids. Step les utilise pour actualiser la couche.

In [ ]:
opt_b=torch.optim.SGD(layer.parameters(),lr=.1)
opt_b.zero_grad();loss_before=criterion(layer(xb),yb)
weights_before=layer.weight.detach().clone()
loss_before.backward()
display(pd.DataFrame(layer.weight.grad.numpy(),columns=['gradient_poids_x1','gradient_poids_x2']))
assert torch.equal(layer.weight.detach(),weights_before)
opt_b.step()
loss_after=criterion(layer(xb),yb).item()
print('Poids après step :');display(pd.DataFrame(layer.weight.detach().numpy()))
print('Perte avant :',loss_before.item(),'après :',loss_after)
assert loss_after<loss_before.item()

## Pourquoi ajouter une non-linéarité

XOR : une droite ne peut pas isoler les deux classes alternées. La démonstration évalue les quatre lignes train uniquement.

In [ ]:
# XOR : illustration de la non-linéarité, évaluée sur ses quatre lignes d’apprentissage.
xx=torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
yx=torch.tensor([0,1,1,0])
torch.manual_seed(2026)
net_xor=nn.Sequential(nn.Linear(2,8),nn.ReLU(),nn.Linear(8,2))
opt_xor=torch.optim.Adam(net_xor.parameters(),lr=.05)
for _ in range(300):
 opt_xor.zero_grad();lx=nn.CrossEntropyLoss()(net_xor(xx),yx);lx.backward();opt_xor.step()
net_xor.eval()
with torch.no_grad():px=torch.softmax(net_xor(xx),dim=1)[:,1]
display(pd.DataFrame({'x1':xx[:,0],'x2':xx[:,1],'classe':yx,'probabilite':px}))
print('Paramètres :',sum(p.numel() for p in net_xor.parameters()))
print('Accuracy sur les quatre lignes train :',((px>=.5).long()==yx).float().mean().item())
assert ((px>=.5).long()==yx).all()
print('Ce calcul illustre une fonction non linéaire. Il ne démontre aucune généralisation Kaggle.')

## Les transformations numériques

Le prétraitement est un pipeline ajusté dans chaque fit. Il ne consulte pas les réponses de validation.

In [ ]:
def prepare(train_x):
 # Prétraitement du modèle linéaire J2, réutilisé en dense pour le réseau.
 return clone(make_model('fraud',tr,'linear')[:-1])
def dense(a):
 return np.asarray(a.toarray() if hasattr(a,'toarray') else a,dtype=np.float32)
print('Prétraitement défini : imputation, mise à l’échelle et catégories apprises dans chaque train de fold.')

## Les dimensions et la référence constante

Avant d’entraîner : vérifier les formes, le type, la mémoire et la performance sans information.

In [ ]:
# Prétraitement ajusté sur le train du premier fold uniquement.
pp=prepare(X)
a,b=folds[0]
arr_train=dense(pp.fit_transform(X.iloc[a]));arr_valid=dense(pp.transform(X.iloc[b]))
print('Train transformé :',arr_train.shape,'validation transformée :',arr_valid.shape)
print('Type :',arr_train.dtype,'mémoire train des valeurs :',round(arr_train.nbytes/1e6,2),'Mo')
print('Paramètres du réseau tabulaire :',32*arr_train.shape[1]+98)
assert arr_train.shape[1]==arr_valid.shape[1]
# Une référence sans information : prédire la proportion train pour tout le monde.
from sklearn.metrics import log_loss,accuracy_score
prior=float(y.iloc[a].mean())
print('Référence constante : accuracy =',accuracy_score(y.iloc[b],np.zeros(len(b))),'log-loss =',log_loss(y.iloc[b],np.full(len(b),prior)))
print('Beaucoup de paiements normaux : une forte accuracy ne démontre pas une détection utile.')

## La boucle tabulaire complète

Retrouver les cinq opérations déjà observées sur le petit batch. Les options d’early stopping restent désactivées dans la comparaison à huit epochs fixes.

In [ ]:
EPOCHS=8;BATCH=256
class FraudNet(nn.Module):
 def __init__(self,n):
  super().__init__();self.layers=nn.Sequential(nn.Linear(n,32),nn.ReLU(),nn.Linear(32,2))
 def forward(self,x):return self.layers(x)
def train_net(train_x,train_y,lr=.001,eval_x=None,eval_y=None,epochs=EPOCHS,patience=None):
 torch.manual_seed(2026)
 prep=prepare(train_x);a=torch.tensor(dense(prep.fit_transform(train_x)))
 target=torch.tensor(np.asarray(train_y),dtype=torch.long)
 net=FraudNet(a.shape[1]);opt=torch.optim.AdamW(net.parameters(),lr=lr,weight_decay=.01)
 criterion=nn.CrossEntropyLoss();history=[]
 best_loss=float("inf");best_state=None;stale=0;best_epoch=None
 g=torch.Generator().manual_seed(2026)
 b=torch.tensor(dense(prep.transform(eval_x))) if eval_x is not None else None
 for epoch in range(1,epochs+1):
  net.train()
  for idx in torch.randperm(len(a),generator=g).split(BATCH):
   opt.zero_grad();logits=net(a[idx]);loss=criterion(logits,target[idx])
   loss.backward();opt.step()
  net.eval()
  with torch.no_grad():
   train_loss=criterion(net(a),target).item()
   row={'epoch':epoch,'train_loss':train_loss}
   if b is not None:
    out=net(b);row['valid_loss']=criterion(out,torch.tensor(np.asarray(eval_y),dtype=torch.long)).item()
    row['valid_auc']=roc_auc_score(eval_y,torch.softmax(out,1)[:,1].numpy())
  history.append(row)
  if patience is not None:
   if b is None:raise ValueError('Early stopping exige une validation interne explicite')
   if row['valid_loss']<best_loss-1e-5:
    best_loss=row['valid_loss'];best_state={k:v.detach().clone() for k,v in net.state_dict().items()};best_epoch=epoch;stale=0
   else:stale+=1
   if stale>=patience:break
 if patience is not None:
  net.load_state_dict(best_state)
  net.best_epoch=best_epoch
 return prep,net,pd.DataFrame(history)
def predict_net(prep,net,frame):
 net.eval();parts=[]
 with torch.no_grad():
  for start in range(0,len(frame),4096):
   a=torch.tensor(dense(prep.transform(frame.iloc[start:start+4096])))
   parts.append(torch.softmax(net(a),dim=1)[:,1].numpy())
 return np.concatenate(parts)

print("Boucle définie : architecture fixe, CPU, huit epochs par défaut, early stopping optionnel sur validation interne.")

## Les courbes sur un fold

Observer toute la courbe avant de diagnostiquer. La perte validation de ce fold ne constitue pas le contrôle externe.

In [ ]:
a,b=folds[0]
prep_demo,net_demo,history=train_net(X.iloc[a],y.iloc[a],eval_x=X.iloc[b],eval_y=y.iloc[b])
display(history.round(5))
ax=history.plot(x='epoch',y=['train_loss','valid_loss'],figsize=(10,4));ax.set_ylabel('Cross-entropy');plt.show()
print('La validation de cette courbe est le premier fold. Aucun checkpoint n’est choisi pour la comparaison à huit epochs fixes.')

## train, eval et no_grad

Exemple séparé de dropout pour rendre les différences observables. Le réseau principal ne contient pas de dropout.

In [ ]:
torch.manual_seed(2026)
drop=nn.Dropout(.5);ones=torch.ones(12,requires_grad=True)
drop.train();u=drop(ones);v=drop(ones)
print('train :',u.detach().tolist());print('train autre passage :',v.detach().tolist())
drop.eval();e=drop(ones)*1
print('eval :',e.detach().tolist(),'requires_grad =',e.requires_grad)
with torch.no_grad():ng=drop(ones)*1
print('eval + no_grad : requires_grad =',ng.requires_grad)
assert e.requires_grad and not ng.requires_grad
print('Dropout n’est pas utilisé dans le réseau principal de cet atelier. AdamW y utilise weight_decay=0,01.')

## Deux learning rates, trois folds

Six entraînements neuronaux et trois fits logistiques. Les deux learning rates utilisent les mêmes données, huit epochs et la même architecture.

In [ ]:
network_rows=[]
for lr in [.001,.01]:
 scores=[];start=time.perf_counter()
 for a,b in folds:
  pp,net,h=train_net(X.iloc[a],y.iloc[a],lr=lr)
  scores.append(roc_auc_score(y.iloc[b],predict_net(pp,net,X.iloc[b])))
 network_rows.append({'modele':'reseau','lr':lr,'auc_cv':np.mean(scores),'ecart_type':np.std(scores),'secondes':time.perf_counter()-start})
network_table=pd.DataFrame(network_rows)
base=make_model('fraud',tr,'linear',params={'C':.1})
start=time.perf_counter();res=cross_validate(base,X,y,cv=folds,scoring='roc_auc',n_jobs=1)
base_row={'modele':'logistique_C01','lr':np.nan,'auc_cv':res['test_score'].mean(),'ecart_type':res['test_score'].std(),'secondes':time.perf_counter()-start}
comparison=pd.concat([network_table,pd.DataFrame([base_row])],ignore_index=True)
display(comparison.round(5));comparison.to_csv(RUN/'comparaison.csv',index=False)
best_lr=float(network_table.sort_values('auc_cv',ascending=False).iloc[0].lr)
print('Meilleur learning rate du réseau :',best_lr,'; le réseau ne gagne pas nécessairement.')

## Un early stopping explicite

Split interne distinct du contrôle J2. Règle fixée : perte validation, au maximum 40 epochs et patience trois. Démonstration séparée, pas un nouveau score de tournoi.

In [ ]:
from sklearn.model_selection import train_test_split
inner_a,inner_b=train_test_split(np.arange(len(X)),test_size=.25,stratify=y,random_state=2027)
pp_stop,net_stop,history_stop=train_net(X.iloc[inner_a],y.iloc[inner_a],lr=.01,eval_x=X.iloc[inner_b],eval_y=y.iloc[inner_b],epochs=40,patience=3)
print('Epochs réellement parcourues :',len(history_stop),'checkpoint retenu :',net_stop.best_epoch)
display(history_stop.tail(6).round(5))
history_stop.plot(x='epoch',y=['train_loss','valid_loss'],figsize=(10,4));plt.show()
assert net_stop.best_epoch<=len(history_stop)<=40
print('Démonstration séparée sur split interne. Ce réseau ne remplace pas le candidat comparé à huit epochs fixes.')

## Le contrôle après le choix

Le réseau retenu pour le tournoi garde huit epochs fixes. Le contrôle J2 est déjà connu et sert au diagnostic.

In [ ]:
prep_selected,net_selected,history_selected=train_net(X,y,lr=best_lr)
pred_valid=predict_net(prep_selected,net_selected,Xv)
control_auc=roc_auc_score(yv,pred_valid)
print('Réseau retenu parmi les deux learning rates : AUC contrôle =',round(control_auc,6))
print('Perte et AUC répondent à deux questions différentes ; aucune calibration démontrée.')
prior=ROOT/'runs/J3/tournoi.csv'
if prior.exists():
 print('Comparaison J3 (vérifier que le manifeste et le sous-échantillon correspondent) :');display(pd.read_csv(prior))
else:print('Tableau J3 absent : utiliser les résultats projetés et vérifier le protocole avant comparaison.')

## Exporter un candidat complet

Conserver poids, prétraitement et architecture, puis vérifier le CSV officiel.

In [ ]:
from sklearn.model_selection import train_test_split
from mllab.competition import read_competition
all_train,test,sample=read_competition('fraud',root=ROOT)
fit_train,_=train_test_split(all_train,train_size=LIMIT,stratify=all_train.is_fraud,random_state=2026)
prep_final,net_final,history_final=train_net(xy(fit_train,'fraud'),fit_train.is_fraud.astype(int),lr=best_lr)
pred_test=predict_net(prep_final,net_final,xy(test,'fraud'))
submission=sample.copy();submission['is_fraud']=pred_test
assert list(submission)==['id','is_fraud'] and np.array_equal(submission.id,test.id)
assert np.isfinite(pred_test).all() and ((pred_test>=0)&(pred_test<=1)).all()
submission.to_csv(RUN/'submission_J4.csv',index=False)
import joblib
joblib.dump(prep_final,RUN/'pretraitement.joblib')
torch.save({'state_dict':net_final.state_dict(),'n_inputs':net_final.layers[0].in_features,'lr':best_lr,'epochs':EPOCHS},RUN/'poids_reseau.pt')
manifest={'train_ids_sha256':ids_sha,'donnees_sha256':protocol_record['data_sha256'],'epochs':EPOCHS,'batch':BATCH,'lr':best_lr,'torch':torch.__version__,'appareil':'cpu','controle_auc':control_auc,'lignes_submission':len(submission),'submission_sha256':hashlib.sha256((RUN/'submission_J4.csv').read_bytes()).hexdigest(),'score_public':None,'note':'Conserver les poids, le prétraitement et l’architecture ensemble.'}
(RUN/'manifest.json').write_text(json.dumps(manifest,indent=2))
history.to_csv(RUN/'courbe_demo.csv',index=False)
display(submission.head());print(len(submission),'lignes exportées ; aucun dépôt automatique.')

## Vérifier le retour des artefacts

Recharger les artefacts produits et vérifier les vingt premières prédictions. Ne pas confondre une sauvegarde de poids et un système autonome.

In [ ]:
# Réutiliser les artefacts locaux que nous venons de produire.
loaded_pp=joblib.load(RUN/'pretraitement.joblib')
loaded_checkpoint=torch.load(RUN/'poids_reseau.pt',map_location='cpu',weights_only=True)
loaded_net=FraudNet(loaded_checkpoint['n_inputs']);loaded_net.load_state_dict(loaded_checkpoint['state_dict'])
p_check=predict_net(loaded_pp,loaded_net,xy(test.head(20),'fraud'))
assert np.allclose(p_check,pred_test[:20],rtol=1e-6,atol=1e-8)
print('20 prédictions identiques après sauvegarde et rechargement du prétraitement et des poids.')
print('Comparer local/Kaggle : mêmes fichiers officiels, mêmes folds, versions notées. Le temps seul peut changer.')

## Le journal et la décision

Choisir le réseau ou la référence sur les preuves et expliquer une prochaine expérience limitée.

In [ ]:
journal={'hypothese':'À compléter : effet attendu du learning rate','preuve':'À compléter : comparaison CV, dispersion et durée','decision':'À compléter : garder réseau ou référence, et pourquoi','prochaine_experience':'Un changement, mêmes folds, budget explicite','score_public_reel':None}
(RUN/'journal.json').write_text(json.dumps(journal,indent=2,ensure_ascii=False))
print('Boucle : zero_grad → forward → loss → backward → step.')
print('Fichiers :',', '.join(sorted(p.name for p in RUN.iterdir() if p.is_file())))

## Remise

Conserver les tableaux, le manifeste et le CSV. Compléter le journal avec une hypothèse, une preuve chiffrée, une décision et une prochaine expérience bornée. Préparer une explication orale de deux minutes. Le contrôle local est déjà connu depuis J2. Le test officiel n’a pas de cible accessible et aucun score public n’est inventé.